## Section 1 - Introduction

This tool now has four capabilities for exploring drug-target interactions:
- Search drug and target interactions by compound or protein name
- Retrieve FDA drug safety and adverse event data
- Visualize interaction networks as weighted graphs
- Scan for drug repurposing candidates by clinical phase

In [ ]:
## Section 2 - Setup

from src.chembl_query import search_target, get_bioactivities, search_molecule
from src.pubchem_query import get_compound_properties, get_compound_image_url
from src.analyzer import analyze, format_results
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
## Section 3 - Target Search Example

# Search for EGFR as a target
results = analyze("EGFR", mode="target")

# Show raw results as a pandas DataFrame
if results.get("results"):
    df = pd.DataFrame(results["results"])
    display(df.head())
else:
    print("No results found")

In [ ]:
## Section 4 - Drug Search Example

# Search for Imatinib as a drug
drug_results = analyze("Imatinib", mode="drug")

# Show compound properties
print("ChEMBL Info:", drug_results.get("chembl_info"))
print("PubChem Info:", drug_results.get("pubchem_info"))

In [ ]:
## Section 5 - Visualization

# Take the bioactivity results for EGFR
egfr_results = analyze("EGFR", mode="target")
bio_data = egfr_results.get("results", [])

if bio_data:
    # Filter for valid IC50 values
    valid_data = [d for d in bio_data if d.get("ic50_nm") is not None]
    if valid_data:
        # Limit to top 10 for plotting
        top_data = valid_data[:10]
        names = [d.get("drug_name", "Unknown")[:15] for d in top_data]
        ic50_values = [float(d.get("ic50_nm")) for d in top_data]
        
        plt.figure(figsize=(10, 6))
        plt.bar(names, ic50_values, color='skyblue')
        plt.xlabel('Molecule ChEMBL ID')
        plt.ylabel('IC50 in nM')
        plt.title('Top EGFR Inhibitors by IC50 (ChEMBL data)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No valid IC50 data for plotting")
else:
    print("No bioactivity results for EGFR")

## FDA Safety Data

OpenFDA is a free public database maintained by the US Food and Drug Administration that provides access to drug labels, adverse event reports, and other safety information. It helps researchers understand how drugs behave in real patients outside of controlled laboratory settings.

Drug safety data matters when studying drug-target interactions because a compound might bind strongly to a target in the lab but still cause harmful side effects in people. Combining interaction data with safety reports gives a more complete picture of whether a drug is actually useful and safe for treatment.

In [ ]:
from src.openfda_query import get_drug_label_summary
summary = get_drug_label_summary("Imatinib")
print("Drug Interactions:")
print(summary.get("interactions_text", "None"))
print("\nTop Adverse Events:")
for event in summary.get("adverse_events", []):
    print(f"  {event['reaction_name']} - Count: {event['count']}, Serious: {event['serious']}")

## Drug-Target Network Graph

The network graph shows drugs and proteins as connected points, with lines representing interactions between them. The thickness of each line shows how strongly the drug binds to the protein, making it easy to see which connections are most potent.

Visualizing these interactions as a network is useful because it reveals patterns that tables alone might miss, like when one drug hits multiple targets or when several drugs target the same protein. This helps researchers spot new uses for existing drugs and understand side effects.

In [ ]:
from src.chembl_query import search_target, get_bioactivities
from src.visualizer import build_interaction_network, summarize_network

results = search_target("EGFR")
if results:
    target_id = results[0]["target_chembl_id"]
    bioactivities = get_bioactivities(target_id, limit=15)
    print(f"Found {len(bioactivities)} bioactivities for EGFR")

In [ ]:
summary = summarize_network("EGFR", bioactivities)
import pandas as pd
print(pd.Series(summary).to_string())

In [ ]:
build_interaction_network("EGFR", bioactivities,
      output_file="egfr_network.png")
from IPython.display import Image
Image("egfr_network.png")

## Drug Repurposing Scanner

Drug repurposing means finding new medical uses for drugs that are already approved or in clinical trials. This approach matters for drug discovery because it can save years of development time and billions of dollars, since the safety and manufacturing data for these drugs already exists.

Keep in mind that these computational predictions are just starting points for research and do not replace actual clinical testing or medical conclusions.

In [ ]:
from src.repurposing import find_repurposing_candidates
from src.repurposing import format_repurposing_results, get_phase_label

candidates = find_repurposing_candidates("EGFR", min_phase=2)
print(f"Found {len(candidates)} repurposing candidates")

In [ ]:
print(format_repurposing_results(candidates))

In [ ]:
import pandas as pd
if candidates:
    df = pd.DataFrame(candidates)
    df["Phase Label"] = df["max_phase"].apply(get_phase_label)
    display(df[["pref_name", "Phase Label", "pchembl_value",
                   "ic50_nM", "molecule_chembl_id"]])